In [ ]:
# Scanner fingerprint table for the RSNA knee training set.
#
# Why: random K-fold is reported to inflate AUC by ~0.053 on this data through scanner
# memorisation. If our folds do not group by scanner, our offline numbers are optimistic and every
# model selection made on them was made against the wrong target. We currently have NO trustworthy
# offline gate (gold-58 is retired), which is the real blocker on the accuracy axis - not compute.
# This reads one header per series and writes the fingerprints needed to build a grouped split.
#
# CPU only, no GPU quota spent. Header-only reads: no pixel data is touched.
import os, glob, csv, time
from concurrent.futures import ThreadPoolExecutor
import pydicom

TAGS = ['Manufacturer', 'ManufacturerModelName', 'SoftwareVersions', 'ImagingFrequency',
        'ReceiveCoilName', 'StationName', 'MagneticFieldStrength', 'Laterality',
        'ImageLaterality', 'PatientID', 'PatientSex', 'PatientAge', 'SeriesDescription',
        'ScanningSequence', 'SequenceName', 'RepetitionTime', 'EchoTime',
        'SliceThickness', 'SpacingBetweenSlices', 'PixelSpacing', 'Rows', 'Columns']


def find_root():
    for b in ["/kaggle/input/competitions/rsna-knee-abnormality-detection",
              "/kaggle/input/rsna-knee-abnormality-detection"]:
        if os.path.exists(b + "/train.csv"):
            return b
    for d, _, f in os.walk("/kaggle/input"):
        if "train.csv" in f:
            return d
    raise RuntimeError("no train root")


ROOT = find_root()
SDIR = ROOT + "/train_series"
if not os.path.isdir(SDIR):
    SDIR = ROOT + "/train_images"
print("root:", ROOT, "| series dir:", SDIR, flush=True)

series = []
for study in sorted(os.listdir(SDIR)):
    sp = os.path.join(SDIR, study)
    if not os.path.isdir(sp):
        continue
    for ser in sorted(os.listdir(sp)):
        series.append((study, ser, os.path.join(sp, ser)))
print("series found:", len(series), "| studies:", len({s[0] for s in series}), flush=True)


def one(rec):
    study, ser, path = rec
    row = {"StudyInstanceUID": study, "SeriesInstanceUID": ser}
    try:
        fs = [e.name for e in os.scandir(path) if e.name.endswith(".dcm")]
        if not fs:
            return row
        row["n_slices"] = len(fs)
        ds = pydicom.dcmread(os.path.join(path, sorted(fs)[len(fs) // 2]),
                             stop_before_pixels=True, force=True, specific_tags=TAGS)
        for t in TAGS:
            v = getattr(ds, t, None)
            if v is not None:
                row[t] = str(v).strip()
    except Exception as e:
        row["error"] = type(e).__name__
    return row


t0 = time.time()
with ThreadPoolExecutor(max_workers=32) as ex:
    rows = list(ex.map(one, series))
print("read %d series in %.1fs" % (len(rows), time.time() - t0), flush=True)

cols = ["StudyInstanceUID", "SeriesInstanceUID", "n_slices"] + TAGS + ["error"]
out = "/kaggle/working/train_scanner_headers.csv"
with open(out, "w", newline="", encoding="utf-8") as fh:
    w = csv.DictWriter(fh, fieldnames=cols, extrasaction="ignore")
    w.writeheader()
    for r in rows:
        w.writerow(r)
print("wrote", out, "rows", len(rows), flush=True)

# quick read on how much grouping actually matters
from collections import Counter
fp = Counter()
for r in rows:
    fp[(r.get("Manufacturer", ""), r.get("ManufacturerModelName", ""),
        r.get("SoftwareVersions", ""), r.get("StationName", ""))] += 1
print("distinct scanner fingerprints:", len(fp), flush=True)
for k, v in fp.most_common(12):
    print("  %5d  %s" % (v, " | ".join(x[:28] for x in k)), flush=True)
pid = Counter(r.get("PatientID", "") for r in rows if r.get("PatientID"))
rep = sum(1 for _, v in pid.items() if v > 1)
print("distinct PatientIDs: %d | ids appearing in >1 series: %d" % (len(pid), rep), flush=True)
